In [1]:
import time
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ============================================================================
# 1. Modellaufbau: Chunking, Embedding, Retrieval, LLM
# ============================================================================
print("Lade Daten für Modell...")
with open('../../Dataset/goldenData.pkl', 'rb') as file:
    df = pickle.load(file)

# Chunking
chunks = []
metadaten = []
for idx, row in df.iterrows():
    abstract = row['abstract']
    for satz in abstract.split('. '):
        if satz:
            chunks.append(satz.strip())
            metadaten.append({
                'pmcid': row['pmcid'],
                'title': row['title'],
                'year': row['year'],
                'chunk_index': len([c for c in chunks if c.endswith(satz)])   # vereinfacht
            })

print(f"Erstellt {len(chunks)} Chunks.")

# Embedding
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embed_model.encode(chunks)

def retrieve_with_metadata(query, k=2):
    """Sicherer Retrieval mit Begrenzung auf verfügbare Chunks."""
    k = min(k, len(chunks))                     # Verhindert IndexError
    query_emb = embed_model.encode([query])[0]
    similarities = np.dot(chunk_embeddings, query_emb) / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_emb)
    )
    top_k_idx = np.argsort(similarities)[-k:][::-1]
    result_chunks = [chunks[i] for i in top_k_idx]
    result_metadata = [metadaten[i] for i in top_k_idx]
    return result_chunks, result_metadata

# LLM laden (FLAN-T5-large)
print("Lade LLM...")
llm_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(llm_name)

def generate_answer(query):
    """Generiert 'yes', 'no' oder 'maybe' basierend auf den 10 ähnlichsten Chunks."""
    gefundene, _ = retrieve_with_metadata(query, k=10)
    kontext = " ".join(gefundene)
    prompt = f"""Answer with exactly one word: 'yes', 'no', or 'maybe'.
Based on the given context, answer the question.
Context: {kontext}
Question: {query}
Answer:"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = llm.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        num_beams=1
    )
    antwort = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return antwort.strip()

# ============================================================================
# 2. Evaluation mit goldenen Fragen
# ============================================================================
def parse_answer(text: str) -> str:
    text = text.strip().lower()
    if "yes" in text:
        return "yes"
    if "no" in text:
        return "no"
    if "maybe" in text:
        return "maybe"
    return "unknown"

print("\nLade Fragen mit Ground Truth...")
df_questions = pd.read_pickle('../../Dataset/golden69_questions.pkl')

# Spalten prüfen
required_cols = ['question', 'final_decision']
if not all(col in df_questions.columns for col in required_cols):
    raise ValueError(f"Fehlende Spalten in goldenen Fragen: {required_cols}")

# Prüfen, ob eine Spalte für PMCID existiert
if 'pmcid' not in df_questions.columns:
    print("WARNUNG: Spalte 'pmcid' nicht im Fragen-DataFrame. Retrieval-Metriken werden nicht berechnet.")
    has_retrieval_gt = False
else:
    has_retrieval_gt = True

true_labels = []
pred_labels = []
hit_rates = []
mrr_scores = []
latencies = []

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    start = time.time()
    question = row['question']
    true = row['final_decision'].strip().lower()

    # Retrieval (für Metriken und für Generierung)
    chunks_ret, meta_ret = retrieve_with_metadata(question, k=5)
    retrieved_pmcids = [m.get('pmcid') for m in meta_ret] if has_retrieval_gt else []

    # Retrieval-Metriken (nur wenn Ground Truth vorhanden)
    if has_retrieval_gt:
        true_pmcid = row.get('pmcid', None)
        if true_pmcid is not None:
            hit = 1 if true_pmcid in retrieved_pmcids else 0
            hit_rates.append(hit)
            if hit:
                rank = retrieved_pmcids.index(true_pmcid) + 1
                mrr_scores.append(1/rank)
            else:
                mrr_scores.append(0)
        else:
            # Falls Zeile keine pmcid hat, können wir nichts berechnen
            hit_rates.append(0)
            mrr_scores.append(0)

    # Generation
    answer = generate_answer(question)
    pred = parse_answer(answer)

    true_labels.append(true)
    pred_labels.append(pred)

    latencies.append(time.time() - start)

# ============================================================================
# 3. Ausgabe der Metriken
# ============================================================================
# Nur gültige yes/no/maybe für Klassifikationsmetriken
valid_indices = [i for i, p in enumerate(pred_labels) if p in ['yes', 'no', 'maybe']]
true_valid = [true_labels[i] for i in valid_indices]
pred_valid = [pred_labels[i] for i in valid_indices]

accuracy = accuracy_score(true_valid, pred_valid)
labels = ['yes', 'no', 'maybe']
precision, recall, f1, _ = precision_recall_fscore_support(true_valid, pred_valid, labels=labels, average=None)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(true_valid, pred_valid, average='macro')
cm = confusion_matrix(true_valid, pred_valid, labels=labels)

print("\n" + "="*60)
print("EVALUATIONSERGEBNISSE")
print("="*60)
print(f"Anzahl ausgewerteter Fragen: {len(df_questions)}")
print(f"Davon 'unknown'-Antworten (nicht parsebar): {pred_labels.count('unknown')}")
print(f"Gültige Vergleiche (yes/no/maybe): {len(true_valid)}")
print(f"\nAccuracy (über alle gültigen): {accuracy:.4f}")
print("\nKlassenweise Metriken:")
for i, label in enumerate(labels):
    print(f"  {label.upper()}  - Precision: {precision[i]:.4f}, Recall: {recall[i]:.4f}, F1: {f1[i]:.4f}")
print(f"\nMakro-Durchschnitt (ungewichtet): Precision: {macro_precision:.4f}, Recall: {macro_recall:.4f}, F1: {macro_f1:.4f}")
print("\nConfusion Matrix (Zeilen = wahr, Spalten = vorhergesagt):")
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)

# Retrieval-Metriken (nur wenn vorhanden)
if has_retrieval_gt and hit_rates:
    print(f"\nRetrieval-Metriken (k=5):")
    print(f"  Hit Rate@5: {np.mean(hit_rates):.4f}")
    print(f"  Mean Reciprocal Rank (MRR): {np.mean(mrr_scores):.4f}")
else:
    print("\nRetrieval-Metriken wurden nicht berechnet (keine 'pmcid' in goldenen Fragen).")

print(f"\nDurchschnittliche Latenz pro Frage: {np.mean(latencies):.2f} Sekunden")

# Optional: Ergebnisse speichern
df_results = pd.DataFrame({
    'question': df_questions['question'],
    'true_label': true_labels,
    'pred_label': pred_labels
})
df_results.to_csv('rag_evaluation_results.csv', index=False)
print("\nErgebnisse wurden in 'rag_evaluation_results.csv' gespeichert.")

/mnt/Fastdata/NeueFische_Bootcamp/Capstone1-text-prediction-for-scientific-papers/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Lade Daten für Modell...
Erstellt 900 Chunks.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 9287.61it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Lade LLM...


Loading weights: 100%|██████████| 558/558 [00:00<00:00, 7870.88it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Lade Fragen mit Ground Truth...
WARNUNG: Spalte 'pmcid' nicht im Fragen-DataFrame. Retrieval-Metriken werden nicht berechnet.


 28%|██▊       | 19/69 [00:16<00:42,  1.17it/s]


KeyboardInterrupt: 

In [2]:
df_test = pd.read_pickle('../../Dataset/golden69_questions.pkl')

In [3]:
df_test.head()

,pubid,question,context,long_answer,final_decision,Accession ID
1,21569408,Does context matter for the relationship betwe...,{'contexts': ['A growing body of research emph...,The homogeneity we found in the deprivation-mo...,no,3103414
2,22720085,Does insulin resistance drive the association ...,{'contexts': ['Several studies have shown asso...,The association between plasma glucose levels ...,yes,3376119
3,20602784,Identification of racial disparities in breast...,{'contexts': ['This paper investigates the imp...,This study investigates both relative and abso...,yes,2910022
4,21658267,"Do improvements in outreach, clinical, and fam...",{'contexts': ['There are three main service de...,Our findings support the importance of increas...,yes,3141447
5,26163474,Is there a connection between sublingual varic...,{'contexts': ['Sublingual varices have earlier...,An association was found between sublingual va...,yes,4499223


In [ ]:
import time
import pickle
import numpy as np
import pandas as pd
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# ============================================================================
# 1. Modellaufbau: Chunking, Embedding, Retrieval, LLM
# ============================================================================
print("Lade Daten für Modell...")
with open('../../Dataset/golden10K_data.pkl', 'rb') as file:
    df = pickle.load(file)

# Chunking
chunks = []
metadaten = []
for idx, row in df.iterrows():
    abstract = row['abstract']
    for satz in abstract.split('. '):
        if satz:
            chunks.append(satz.strip())
            metadaten.append({
                'pmcid': row['pmcid'],          # interne PMCID für Retrieval
                'title': row['title'],
                'year': row['year'],
                'chunk_index': len([c for c in chunks if c.endswith(satz)])
            })

print(f"Erstellt {len(chunks)} Chunks.")

# Embedding
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
chunk_embeddings = embed_model.encode(chunks)

def retrieve_with_metadata(query, k=2):
    k = min(k, len(chunks))
    query_emb = embed_model.encode([query])[0]
    similarities = np.dot(chunk_embeddings, query_emb) / (
        np.linalg.norm(chunk_embeddings, axis=1) * np.linalg.norm(query_emb)
    )
    top_k_idx = np.argsort(similarities)[-k:][::-1]
    result_chunks = [chunks[i] for i in top_k_idx]
    result_metadata = [metadaten[i] for i in top_k_idx]
    return result_chunks, result_metadata

# LLM laden (FLAN-T5-large)
print("Lade LLM...")
llm_name = "google/flan-t5-large"
tokenizer = AutoTokenizer.from_pretrained(llm_name)
llm = AutoModelForSeq2SeqLM.from_pretrained(llm_name)

def generate_answer(query):
    gefundene, _ = retrieve_with_metadata(query, k=10)
    kontext = " ".join(gefundene)
    prompt = f"""Answer with exactly one word: 'yes', 'no', or 'maybe'.
Based on the given context, answer the question.
Context: {kontext}
Question: {query}
Answer:"""
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=512)
    outputs = llm.generate(
        **inputs,
        max_new_tokens=10,
        do_sample=False,
        num_beams=1
    )
    antwort = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return antwort.strip()

# ============================================================================
# 1. Evaluation mit goldenen Fragen
# ============================================================================
def parse_answer(text: str) -> str:
    text = text.strip().lower()
    if "yes" in text:
        return "yes"
    if "no" in text:
        return "no"
    if "maybe" in text:
        return "maybe"
    return "unknown"

print("\nLade Fragen mit Ground Truth...")
df_questions = pd.read_pickle('../../Dataset/golden69_questions.pkl')

# Spalten prüfen
required_cols = ['question', 'final_decision']
if not all(col in df_questions.columns for col in required_cols):
    raise ValueError(f"Fehlende Spalten in goldenen Fragen: {required_cols}")

# Prüfen, ob die Spalte für die PMCID existiert (hier 'Accession ID')
if 'Accession ID' not in df_questions.columns:
    print("WARNUNG: Spalte 'Accession ID' nicht im Fragen-DataFrame. Retrieval-Metriken werden nicht berechnet.")
    has_retrieval_gt = False
else:
    has_retrieval_gt = True

true_labels = []
pred_labels = []
hit_rates = []
mrr_scores = []
latencies = []

for _, row in tqdm(df_questions.iterrows(), total=len(df_questions)):
    start = time.time()
    question = row['question']
    true = row['final_decision'].strip().lower()

    # Retrieval
    chunks_ret, meta_ret = retrieve_with_metadata(question, k=5)
    retrieved_pmcids = [m.get('pmcid') for m in meta_ret] if has_retrieval_gt else []

    # Retrieval-Metriken (nur wenn Ground Truth vorhanden)
    if has_retrieval_gt:
        true_accid = row.get('Accession ID', None)   # z.B. "PMC8005659"
        if true_accid is not None:
            hit = 1 if true_accid in retrieved_pmcids else 0
            hit_rates.append(hit)
            if hit:
                rank = retrieved_pmcids.index(true_accid) + 1
                mrr_scores.append(1/rank)
            else:
                mrr_scores.append(0)
        else:
            # Falls Zeile keine Accession ID hat
            hit_rates.append(0)
            mrr_scores.append(0)

    # Generation
    answer = generate_answer(question)
    pred = parse_answer(answer)

    true_labels.append(true)
    pred_labels.append(pred)

    latencies.append(time.time() - start)

# ============================================================================
# 2. Ausgabe der Metriken
# ============================================================================
valid_indices = [i for i, p in enumerate(pred_labels) if p in ['yes', 'no', 'maybe']]
true_valid = [true_labels[i] for i in valid_indices]
pred_valid = [pred_labels[i] for i in valid_indices]

accuracy = accuracy_score(true_valid, pred_valid)
labels = ['yes', 'no', 'maybe']
precision, recall, f1, _ = precision_recall_fscore_support(true_valid, pred_valid, labels=labels, average=None)
macro_precision, macro_recall, macro_f1, _ = precision_recall_fscore_support(true_valid, pred_valid, average='macro')
cm = confusion_matrix(true_valid, pred_valid, labels=labels)

print("\n" + "="*60)
print("EVALUATIONSERGEBNISSE")
print("="*60)
print(f"Anzahl ausgewerteter Fragen: {len(df_questions)}")
print(f"Davon 'unknown'-Antworten (nicht parsebar): {pred_labels.count('unknown')}")
print(f"Gültige Vergleiche (yes/no/maybe): {len(true_valid)}")
print(f"\nAccuracy (über alle gültigen): {accuracy:.4f}")
print("\nKlassenweise Metriken:")
for i, label in enumerate(labels):
    print(f"  {label.upper()}  - Precision: {precision[i]:.4f}, Recall: {recall[i]:.4f}, F1: {f1[i]:.4f}")
print(f"\nMakro-Durchschnitt (ungewichtet): Precision: {macro_precision:.4f}, Recall: {macro_recall:.4f}, F1: {macro_f1:.4f}")
print("\nConfusion Matrix (Zeilen = wahr, Spalten = vorhergesagt):")
cm_df = pd.DataFrame(cm, index=labels, columns=labels)
print(cm_df)

if has_retrieval_gt and hit_rates:
    print(f"\nRetrieval-Metriken (k=5):")
    print(f"  Hit Rate@5: {np.mean(hit_rates):.4f}")
    print(f"  Mean Reciprocal Rank (MRR): {np.mean(mrr_scores):.4f}")
else:
    print("\nRetrieval-Metriken wurden nicht berechnet (keine 'Accession ID' in goldenen Fragen).")

print(f"\nDurchschnittliche Latenz pro Frage: {np.mean(latencies):.2f} Sekunden")

df_results = pd.DataFrame({
    'question': df_questions['question'],
    'true_label': true_labels,
    'pred_label': pred_labels
})
df_results.to_csv('rag_evaluation_results.csv', index=False)
print("\nErgebnisse wurden in 'rag_evaluation_results.csv' gespeichert.")

Lade Daten für Modell...
Erstellt 125246 Chunks.


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10066.02it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Lade LLM...


Loading weights: 100%|██████████| 558/558 [00:00<00:00, 9306.18it/s]
The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning



Lade Fragen mit Ground Truth...


100%|██████████| 69/69 [00:54<00:00,  1.27it/s]


EVALUATIONSERGEBNISSE
Anzahl ausgewerteter Fragen: 69
Davon 'unknown'-Antworten (nicht parsebar): 0
Gültige Vergleiche (yes/no/maybe): 69

Accuracy (über alle gültigen): 0.7391

Klassenweise Metriken:
  YES  - Precision: 0.7556, Recall: 0.8947, F1: 0.8193
  NO  - Precision: 0.7083, Recall: 0.7083, F1: 0.7083
  MAYBE  - Precision: 0.0000, Recall: 0.0000, F1: 0.0000

Makro-Durchschnitt (ungewichtet): Precision: 0.4880, Recall: 0.5344, F1: 0.5092

Confusion Matrix (Zeilen = wahr, Spalten = vorhergesagt):
       yes  no  maybe
yes     34   4      0
no       7  17      0
maybe    4   3      0

Retrieval-Metriken (k=5):
  Hit Rate@5: 0.9855
  Mean Reciprocal Rank (MRR): 0.9589

Durchschnittliche Latenz pro Frage: 0.79 Sekunden

Ergebnisse wurden in 'rag_evaluation_results.csv' gespeichert.



/mnt/Fastdata/NeueFische_Bootcamp/Capstone1-text-prediction-for-scientific-papers/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/mnt/Fastdata/NeueFische_Bootcamp/Capstone1-text-prediction-for-scientific-papers/.venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
